# 03. DCA-Trie v2: Step-Wise Dynamic Expansion

**Purpose:** Implement and evaluate DCA-Trie v2 — step-wise semantic expansion during decoding.

**Method:** Instead of building the full trie upfront, start with a minimal trie containing only first-hop edges. As each entity is committed during generation, score its KG neighbors against `(question + partial_generation)` and expand the trie with only the relevant ones.

**Reference:** Thesis §3.7, Algorithm 2. See `EXPLAINER.md` for background.

**Key difference from v1:** v1 filters paths once before decoding (using only the question). v2 rescopes the trie at each step using both the question AND what has been generated so far.

In [ ]:
# @title 1. Install & Setup
import sys, os, torch, json, copy
from tqdm import tqdm
from datasets import load_dataset
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
has_a100 = "A100" in gpu_name
print(f"GPU: {gpu_name}  |  VRAM: {gpu_mem:.1f} GB")

!pip install -q transformers==4.44 accelerate peft deepspeed tiktoken datasets python-dotenv marisa-trie scikit-learn bitsandbytes trl sentencepiece protobuf wandb networkx sentence-transformers
if has_a100:
    !pip install -q flash-attn --no-build-isolation

if not os.path.exists("graph-constrained-reasoning"):
    !git clone https://github.com/rmanluo/graph-constrained-reasoning.git
%cd graph-constrained-reasoning
sys.path.insert(0, '.')
from src.llms import get_registed_model
from src.trie import MarisaTrie
from sentence_transformers import SentenceTransformer

In [ ]:
# @title 2. Configuration
MODEL_PATH = "rmanluo/GCR-Meta-Llama-3.1-8B-Instruct"
MODEL_NAME = "gcr"
# os.environ["HF_TOKEN"] = "hf_your_token_here"

DATA_PATH = "rmanluo"
DATASET = "RoG-webqsp"
SPLIT = "test"

# DCA-Trie v2 parameters
TAU = 0.25                    # admission threshold τ
ENCODER_NAME = "all-MiniLM-L6-v2"
ENCODER_DEVICE = "cpu"

# Decoding
INDEX_LEN = 2                 # max path length
K = 5
GEN_MODE = "group-beam"
PROMPT_MODE = "zero-shot"
MAX_NEW_TOKENS = 256
ATTN_IMPL = "flash_attention_2" if has_a100 else "sdpa"

PREDICT_PATH = "results/GenPaths"
FORCE = True

tag = f"DCAv2-tau{TAU}-{ENCODER_NAME}"
print(f"τ: {TAU}  |  Dataset: {DATASET}  |  Tag: {tag}")

In [ ]:
# @title 3. Load Models
print("Loading sentence encoder...")
encoder = SentenceTransformer(ENCODER_NAME, device=ENCODER_DEVICE)

import argparse
parser = argparse.ArgumentParser()
LLM = get_registed_model(MODEL_NAME)
LLM.add_args(parser)
args = parser.parse_args([])
args.model_path = MODEL_PATH
args.model_name = MODEL_NAME
args.k = 1  # we'll do step-wise manually, not full beam
args.generation_mode = "greedy"
args.attn_implementation = ATTN_IMPL
args.max_new_tokens = 64
args.maximun_token = 4096

print("Loading LLM...")
model = LLM(args)
model.prepare_for_inference()

import src.utils as utils
import networkx as nx

In [ ]:
# @title 4. DCA-Trie v2: Step-Wise Decoding

def build_path_trie(tokenizer, path_strings):
    """Build a MarisaTrie from a list of path strings."""
    if len(path_strings) == 0:
        return None
    tokenized = tokenizer(path_strings, padding=False, add_special_tokens=False).input_ids
    tokenized = [ids + [tokenizer.eos_token_id] for ids in tokenized]
    return MarisaTrie(tokenized, max_token_id=len(tokenizer) + 1)

def dca_v2_generate(question, start_entities, graph, encoder, llm_model, tokenizer, tau, max_hops=2):
    """
    DCA-Trie v2 step-wise generation.

    Algorithm 2:
      1. Initialize trie with first-hop neighbors of question entities
      2. Generate one token (or one relation-edge step) constrained by trie
      3. When an entity is committed, rescore its KG neighbors against
         (question + partial_generation), expand trie with relevant ones
      4. Repeat until EOS or max_hops reached
    """
    from src.graph_constrained_decoding import GraphConstrainedDecoding

    start_token = "<PATH>"
    end_token = "</PATH>"
    start_id = tokenizer.convert_tokens_to_ids(start_token)
    end_id = tokenizer.convert_tokens_to_ids(end_token)

    # Step 1: Build initial trie from first-hop neighbors
    first_hop_paths = []
    for entity in start_entities:
        if entity not in graph:
            continue
        for neighbor in graph.neighbors(entity):
            rel = graph[entity][neighbor]["relation"]
            path_str = f"{entity} -> {rel} -> {neighbor}"
            first_hop_paths.append(path_str)

    if len(first_hop_paths) == 0:
        return None

    # Score first-hop paths against question
    q_emb = encoder.encode(question, convert_to_numpy=True)
    kept_paths = []
    for p in first_hop_paths:
        p_emb = encoder.encode(p, convert_to_numpy=True)
        score = cosine_similarity(q_emb.reshape(1, -1), p_emb.reshape(1, -1))[0][0]
        if score >= tau:
            kept_paths.append(p)

    if len(kept_paths) == 0:
        return None

    current_trie = build_path_trie(tokenizer, kept_paths)
    if current_trie is None:
        return None

    # Step 2: Step-wise generation loop
    prompt = (
        f"Reasoning path is a sequence of triples in the KG that connects the topic entities "
        f"to answer entities. Given the question, generate reasoning paths starting from "
        f"the topic entities to answer the question.\n\n"
        f"# Question:\n{question}\n"
        f"# Topic entities:\n{', '.join(start_entities)}\n"
    )

    for hop in range(max_hops):
        # Constrained generation for this hop
        llm_input = llm_model.prepare_model_prompt(prompt)
        gcr = GraphConstrainedDecoding(
            tokenizer, current_trie, start_id, end_id, enable_constrained_by_default=False
        )
        inputs = tokenizer(llm_input, return_tensors="pt", add_special_tokens=False)
        input_ids = inputs.input_ids.to(llm_model.model.device)
        attn_mask = inputs.attention_mask.to(llm_model.model.device)

        res = llm_model.model.generate(
            input_ids=input_ids,
            attention_mask=attn_mask,
            generation_config=llm_model.generation_cfg,
            prefix_allowed_tokens_fn=gcr.allowed_tokens_fn,
            return_dict_in_generate=True,
            pad_token_id=tokenizer.eos_token_id,
            max_new_tokens=32,
        )

        output = tokenizer.decode(
            res.sequences[0][input_ids.shape[1]:], skip_special_tokens=True
        )

        # Extract the entity from this hop's output
        # The output looks like: "<PATH>entity -> rel -> entity</PATH> ..."
        # or "entity -> rel -> entity" depending on prompt
        parts = output.split(" -> ")
        if len(parts) >= 3:
            committed_entity = parts[-1].strip()
            # Also check if path contains end token
            if end_token in committed_entity:
                committed_entity = committed_entity.replace(end_token, "").strip()
        else:
            committed_entity = parts[-1].strip() if parts else ""

        # Step 3: If this is the last hop, return
        if hop == max_hops - 1:
            return output

        # Expand trie with scored neighbors of committed entity
        updated = False
        if committed_entity and committed_entity in graph:
            generated_prefix = output.strip()
            context = f"{question} [SEP] {generated_prefix}"
            ctx_emb = encoder.encode(context, convert_to_numpy=True)

            new_paths = []
            for neighbor in graph.neighbors(committed_entity):
                rel = graph[committed_entity][neighbor]["relation"]
                path_str = f"{committed_entity} -> {rel} -> {neighbor}"
                p_emb = encoder.encode(path_str, convert_to_numpy=True)
                score = cosine_similarity(ctx_emb.reshape(1, -1), p_emb.reshape(1, -1))[0][0]
                if score >= tau:
                    new_paths.append(path_str)
                    updated = True

            if updated:
                # Build new expanded trie
                all_paths = list(current_trie.trie.keys()) if hasattr(current_trie, 'trie') else new_paths
                expanded_trie = build_path_trie(tokenizer, new_paths)
                if expanded_trie is not None:
                    current_trie = expanded_trie
                    prompt = prompt + f"\n{output}\n"
                    continue

        # If no expansion, return what we have
        return output

    return None

In [ ]:
# @title 5. Run DCA-Trie v2 Inference
dataset = load_dataset(f"{DATA_PATH}/{DATASET}", split=SPLIT)
print(f"Loaded {len(dataset)} questions")

model_short = MODEL_PATH.split("/")[-1]
postfix = f"{PROMPT_MODE}-{GEN_MODE}-k{K}-index_len{INDEX_LEN}-{tag}"
output_dir = os.path.join(PREDICT_PATH, DATASET, model_short, SPLIT, postfix)
os.makedirs(output_dir, exist_ok=True)
print(f"Output: {output_dir}")

def get_output(path, force):
    if not os.path.exists(path) or force:
        return open(path, "w"), []
    with open(path) as f:
        proc = [json.loads(l)["id"] for l in f]
    return open(path, "a"), proc

fout, processed = get_output(os.path.join(output_dir, "predictions.jsonl"), FORCE)

for data in tqdm(dataset, desc="DCA-Trie v2"):
    qid = data["id"]
    if qid in processed:
        continue

    graph = utils.build_graph(data["graph"])
    truth_paths = utils.get_truth_paths(data["q_entity"], data["a_entity"], graph)
    ground_paths = [utils.path_to_string(p) for p in truth_paths]

    prediction = dca_v2_generate(
        question=data["question"],
        start_entities=data["q_entity"],
        graph=graph,
        encoder=encoder,
        llm_model=model,
        tokenizer=model.tokenizer,
        tau=TAU,
        max_hops=INDEX_LEN,
    )

    if prediction is None:
        continue

    fout.write(json.dumps({
        "id": qid,
        "question": data["question"],
        "prediction": prediction,
        "ground_truth": data["answer"],
        "ground_truth_paths": ground_paths,
        "dca_tau": TAU,
        "dca_version": "v2",
    }) + "\n")
    fout.flush()

fout.close()
print("Done.")